# Notebook 04 — Dataset Creation
### WID2003 Cognitive Science | FSKTM, Universiti Malaya

---

## Overview

Before training a classifier, features and labels must be merged into a single, clean, analysis-ready table. This notebook reshapes the long-format per-task features into a wide participant-level dataset, handles missing data, caps outliers, and applies standardization.

**Inputs**
| File | Description |
|---|---|
| `data/processed/features_per_task.parquet` | Eye-tracking features — one row per participant × task |
| `data/processed/labels.csv` | Performance label per participant |

**Output**
| File | Description |
|---|---|
| `data/processed/dataset_final.parquet` | One row per participant, ~70+ feature columns, ready for modelling |
| `data/processed/scaler.pkl` | Fitted `StandardScaler` (trained on training rows only) |

---

## Learning Objectives

By the end of this notebook, you should be able to:

1. Reshape data from **long format** (participant × task rows) to **wide format** (one row per participant) using `pivot_table`
2. Explain why a `StandardScaler` must be **fit on the training set only** and only applied (transformed) to the test set — and describe what goes wrong if you fit on both
3. Apply **median imputation** for missing values and justify when this is appropriate vs. when it is not
4. Describe the **1.5×IQR rule** for outlier capping and explain why capping is preferred over deletion in small datasets
5. Implement a **stratified train/test split** and explain why stratification matters for imbalanced class labels

---

## Background

### Long-to-Wide Pivoting

After feature extraction, data is stored in **long format**: each row is one (participant, task) pair, and feature columns contain values specific to that task. Machine learning models expect **wide format**: one row per participant with all features as separate columns.

The pivot operation creates column names like:
```
findDice_correct_Total_duration_of_fixations
findDice_correct_aoi_dwell_ratio
whoCheats_correct_Time_to_first_fixation
...
```

This results in approximately **8 tasks × ~15 features = ~120 per-task feature columns**, plus a handful of cross-task aggregate features.

### Missing Data Strategies

Not every participant has data for every AOI on every task (e.g., if they never looked at the answer region). This produces `NaN` values in the wide table.

| Strategy | When to use |
|---|---|
| Median imputation | < 30% missing in a column; distribution is skewed |
| Mean imputation | < 30% missing; distribution is symmetric |
| Drop the column | ≥ 30% missing; too little information to be useful |
| Drop the participant | Participant has nearly all features missing |

We use **median imputation** because many eye-tracking metrics are right-skewed (a few very long fixation durations inflate the mean).

### Outlier Capping (1.5×IQR)

Extreme values can dominate a classifier. The **IQR rule** identifies the interquartile range:

```
lower fence = Q1 − 1.5 × IQR
upper fence = Q3 + 1.5 × IQR
```

Values outside these fences are **clipped** (pulled to the fence value), not deleted. This is preferred in small samples because deletion wastes data.

### Feature Scaling

`StandardScaler` transforms each feature to zero mean and unit variance:

```
z = (x − mean) / std
```

**Critical rule:** fit the scaler on **training rows only**, then apply (`transform`) to both train and test rows. Fitting on all rows leaks test statistics into training — a form of data leakage that artificially inflates cross-validation scores.

### Stratified Train/Test Split

A random split may, by chance, put all High performers in training and all Low performers in test (or vice versa). A **stratified split** ensures the class ratio is preserved in both partitions. With very small N (< 10), there may not be enough data for a hold-out test set — in that case, all participants go into cross-validation and the `split` column is set to `train` for all rows.

---

## Discussion Questions

1. You pivot features from long to wide format, creating ~120 columns from 8 tasks. If a student was absent for one task, approximately how many columns will be `NaN`? How does this affect the imputation strategy?
2. A classmate proposes fitting `StandardScaler` on the full dataset to "get better scaling statistics." Explain specifically what goes wrong with this approach and give a concrete example of how it leads to overly optimistic evaluation results.
3. The 1.5×IQR rule flags outliers based on the distribution of the training data. If a test participant has a genuinely extreme fixation duration (e.g., they stared at the wrong region for 30 seconds), should this be capped? What are the arguments for and against?
4. Cross-task aggregate features (mean fixation duration across all 8 tasks) are computed before the train/test split. Does this introduce data leakage? Justify your answer.
5. With only 20–30 participants, a 20% test set contains 4–6 people. What problems does this cause for evaluating a confusion matrix? What would you do instead?

In [1]:
import sys
sys.path.insert(0, '..')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

from src.config import (
    FEATURES_PER_TASK, LABELS_CSV, DATASET_FINAL, SCALER_PKL,
    PERFORMANCE_LABEL_COL, RANDOM_STATE, TEST_SIZE
)

pd.set_option('display.max_columns', 50)

## 1. Load inputs

In [2]:
features_long = pd.read_parquet(FEATURES_PER_TASK)
labels        = pd.read_csv(LABELS_CSV)

print(f"Features (long): {features_long.shape}")
print(f"Labels:          {labels.shape}")
print(f"Participants in features: {features_long['participant_id'].nunique()}")
print(f"Participants in labels:   {labels['participant_id'].nunique()}")

Features (long): (30, 37)
Labels:          (10, 14)
Participants in features: 4
Participants in labels:   10


## 2. Pivot features from long to wide format

In [3]:
# Identify feature columns (everything except participant_id and task)
feature_cols = [c for c in features_long.columns if c not in ['participant_id', 'task']]

# Pivot: create columns like "findDice_correct_Total_duration_of_fixations"
features_wide = features_long.pivot_table(
    index='participant_id',
    columns='task',
    values=feature_cols,
    aggfunc='mean'  # each participant should have one row per task; mean handles any duplicates
)

# Flatten MultiIndex columns: (feature, task) → "task_feature"
features_wide.columns = [f"{task}_{feat}" for feat, task in features_wide.columns]
features_wide = features_wide.reset_index()

print(f"Wide features shape: {features_wide.shape}")
features_wide.head()

Wide features shape: (4, 265)


,participant_id,findDice_correct_Average_duration_of_Visit,findYummy_correct_Average_duration_of_Visit,frog_correct_Average_duration_of_Visit,frogInBathroom_correct_Average_duration_of_Visit,headphoneInBathroom_correct_Average_duration_of_Visit,spotNeedleInst_correct_Average_duration_of_Visit,whoCheats_correct_Average_duration_of_Visit,whoThief_correct_Average_duration_of_Visit,findDice_correct_Average_duration_of_fixations,findYummy_correct_Average_duration_of_fixations,frog_correct_Average_duration_of_fixations,frogInBathroom_correct_Average_duration_of_fixations,headphoneInBathroom_correct_Average_duration_of_fixations,spotNeedleInst_correct_Average_duration_of_fixations,whoCheats_correct_Average_duration_of_fixations,whoThief_correct_Average_duration_of_fixations,findDice_correct_Average_eye_openness,findYummy_correct_Average_eye_openness,frog_correct_Average_eye_openness,frogInBathroom_correct_Average_eye_openness,headphoneInBathroom_correct_Average_eye_openness,spotNeedleInst_correct_Average_eye_openness,whoCheats_correct_Average_eye_openness,whoThief_correct_Average_eye_openness,...,whoThief_scanpath_length,findDice_total_aoi_visits,findYummy_total_aoi_visits,frog_total_aoi_visits,frogInBathroom_total_aoi_visits,headphoneInBathroom_total_aoi_visits,spotNeedleInst_total_aoi_visits,whoCheats_total_aoi_visits,whoThief_total_aoi_visits,findDice_total_frames,findYummy_total_frames,frog_total_frames,frogInBathroom_total_frames,headphoneInBathroom_total_frames,spotNeedleInst_total_frames,whoCheats_total_frames,whoThief_total_frames,findDice_valid_frames,findYummy_valid_frames,frog_valid_frames,frogInBathroom_valid_frames,headphoneInBathroom_valid_frames,spotNeedleInst_valid_frames,whoCheats_valid_frames,whoThief_valid_frames
0,p001,1233.0,208.0,249.0,350.0,1303.0,NaN,400.0,542.0,1233.0,208.0,249.0,350.0,643.0,NaN,296.0,542.0,9.38149,9.28450,9.92023,9.31388,9.63355,NaN,9.17408,9.51992,...,19402.548338,117.0,120.0,161.0,132.0,122.0,116.0,129.0,131.0,820.0,600.0,2484.0,1260.0,634.0,600.0,600.0,1600.0,583.0,600.0,2456.0,1260.0,600.0,589.0,600.0,1579.0
1,p002,178.0,NaN,510.0,133.0,850.0,NaN,600.0,317.0,178.0,NaN,352.0,133.0,561.0,NaN,238.0,317.0,8.92977,NaN,8.61911,9.35309,9.00642,NaN,8.40178,8.47465,...,10120.742781,92.0,89.0,120.0,101.0,94.0,89.0,96.0,92.0,745.0,609.0,1996.0,1285.0,611.0,603.0,477.0,432.0,601.0,590.0,1959.0,1260.0,600.0,589.0,445.0,393.0
2,test1,1550.0,NaN,550.0,NaN,NaN,1542.0,NaN,NaN,708.0,NaN,550.0,NaN,NaN,1022.0,NaN,NaN,9.28074,NaN,9.13832,NaN,NaN,8.59026,NaN,NaN,...,NaN,43.0,38.0,63.0,46.0,41.0,43.0,NaN,NaN,601.0,301.0,2269.0,1260.0,422.0,600.0,NaN,NaN,590.0,301.0,2202.0,1237.0,421.0,599.0,NaN,NaN
3,test2,162.0,1394.0,NaN,NaN,300.0,NaN,NaN,228.0,162.0,689.0,NaN,NaN,300.0,NaN,NaN,228.0,9.96085,9.43851,NaN,NaN,9.75726,NaN,NaN,9.65798,...,2425.233275,17.0,18.0,17.0,15.0,18.0,15.0,18.0,17.0,194.0,270.0,284.0,142.0,148.0,481.0,294.0,100.0,174.0,239.0,246.0,106.0,128.0,436.0,258.0,98.0


## 3. Cross-task aggregate features

In [4]:
# Mean fixation duration across all tasks
fix_dur_cols = [c for c in features_wide.columns if 'correct_Average_duration_of_fixations' in c]
if fix_dur_cols:
    features_wide['mean_fixation_duration_across_tasks'] = features_wide[fix_dur_cols].mean(axis=1)

# Total correct AOI dwell ratio across tasks
dwell_cols = [c for c in features_wide.columns if 'correct_aoi_dwell_ratio' in c]
if dwell_cols:
    features_wide['total_correct_aoi_dwell_ratio'] = features_wide[dwell_cols].mean(axis=1)

# Mean response time (time to first mouse click on correct AOI) across tasks
rt_cols = [c for c in features_wide.columns if 'correct_Time_to_first_mouse_click' in c]
if rt_cols:
    features_wide['mean_time_to_first_click'] = features_wide[rt_cols].mean(axis=1)

# Mean distractor dwell ratio
distract_cols = [c for c in features_wide.columns if 'distractor_dwell_ratio' in c]
if distract_cols:
    features_wide['mean_distractor_dwell_ratio'] = features_wide[distract_cols].mean(axis=1)

print(f"Shape after aggregate features: {features_wide.shape}")

Shape after aggregate features: (4, 268)


## 4. Merge with labels

In [5]:
dataset = labels.merge(features_wide, on='participant_id', how='left')

assert len(dataset) == len(labels), "Rows dropped during merge — check participant ID alignment"
print(f"Dataset shape: {dataset.shape}")
print(f"Participants: {dataset['participant_id'].nunique()}")

Dataset shape: (10, 281)
Participants: 10


## 5. Missing value analysis and imputation

In [6]:
# Separate feature columns from metadata/label columns
meta_cols = ['participant_id', 'total_score', 'accuracy_rate',
             PERFORMANCE_LABEL_COL, 'speed_label', 'mean_response_time_ms'] + \
            [c for c in dataset.columns if c.endswith('_correct') and len(c) < 30]
meta_cols = [c for c in meta_cols if c in dataset.columns]

X_cols = [c for c in dataset.columns if c not in meta_cols]
n_participants = len(dataset)
print(f"Feature columns before dropping: {len(X_cols)}")
print(f"Participants: {n_participants}")

null_rate = dataset[X_cols].isnull().mean().sort_values(ascending=False)

# With small N, almost every column will appear "mostly missing" under a fixed threshold.
# Only drop columns that are completely empty (100% NaN) — impute everything else.
if n_participants < 20:
    drop_threshold = 1.0   # only drop fully empty columns
    print(f"Small N ({n_participants}) — dropping only 100%-empty columns; imputing the rest.")
else:
    drop_threshold = 0.50
    print(f"Using drop threshold: {drop_threshold:.0%}")

cols_to_drop = null_rate[null_rate >= drop_threshold].index.tolist()
print(f"Dropping {len(cols_to_drop)} columns (≥{drop_threshold:.0%} missing).")
dataset = dataset.drop(columns=cols_to_drop)
X_cols  = [c for c in X_cols if c not in cols_to_drop]

if not X_cols:
    raise ValueError(
        f"All feature columns are completely empty (N={n_participants}). "
        "Check that notebooks 01 and 02 completed successfully."
    )

print(f"Feature columns remaining: {len(X_cols)}")

# Median imputation for remaining columns
imputer = SimpleImputer(strategy='median')
dataset[X_cols] = imputer.fit_transform(dataset[X_cols])

print(f"Remaining nulls after imputation: {dataset[X_cols].isnull().sum().sum()}")


Feature columns before dropping: 267
Participants: 10
Small N (10) — dropping only 100%-empty columns; imputing the rest.
Dropping 267 columns (≥100% missing).


ValueError: All feature columns are completely empty (N=10). Check that notebooks 01 and 02 completed successfully.

## 6. Outlier capping (1.5×IQR)

In [ ]:
def cap_outliers_iqr(df, cols, factor=1.5):
    df = df.copy()
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - factor * IQR
        upper = Q3 + factor * IQR
        df[col] = df[col].clip(lower, upper)
    return df

dataset = cap_outliers_iqr(dataset, X_cols)
print("Outlier capping applied.")

## 7. Train/test split assignment

In [ ]:
N = len(dataset)
print(f"Total participants: {N}")

# Remove participants with missing performance label
dataset_labeled = dataset.dropna(subset=[PERFORMANCE_LABEL_COL]).copy()
print(f"Participants with valid labels: {len(dataset_labeled)}")

if len(dataset_labeled) < 10:
    print("WARNING: Very few participants. All data will be used for cross-validation only — no hold-out test set.")
    dataset_labeled['split'] = 'train'
else:
    train_idx, test_idx = train_test_split(
        dataset_labeled.index,
        test_size=TEST_SIZE,
        stratify=dataset_labeled[PERFORMANCE_LABEL_COL],
        random_state=RANDOM_STATE
    )
    dataset_labeled['split'] = 'train'
    dataset_labeled.loc[test_idx, 'split'] = 'test'

print("\nSplit distribution:")
print(dataset_labeled['split'].value_counts())

## 8. Feature scaling

In [ ]:
train_mask = dataset_labeled['split'] == 'train'

scaler = StandardScaler()
scaler.fit(dataset_labeled.loc[train_mask, X_cols])

# Save unscaled version
dataset_labeled[[c + '_unscaled' for c in X_cols]] = dataset_labeled[X_cols].values

# Apply scaling
dataset_labeled[X_cols] = scaler.transform(dataset_labeled[X_cols])

joblib.dump(scaler, SCALER_PKL)
print(f"Scaler saved: {SCALER_PKL}")

## 9. Summary and save

In [ ]:
print("=== Dataset Summary ===")
print(f"Shape:              {dataset_labeled.shape}")
print(f"Feature columns:    {len(X_cols)}")
print(f"Participants:       {dataset_labeled['participant_id'].nunique()}")
print(f"High performers:    {(dataset_labeled[PERFORMANCE_LABEL_COL] == 1).sum()}")
print(f"Low performers:     {(dataset_labeled[PERFORMANCE_LABEL_COL] == 0).sum()}")
print(f"Train/test:         {dataset_labeled['split'].value_counts().to_dict()}")

dataset_labeled.to_parquet(DATASET_FINAL, index=False)
print(f"\nSaved: {DATASET_FINAL}")